# Análise de Vendedores com Alta Quantidade de Vendas e Baixa Avaliação

Este notebook analisa vendedores da base Olist que possuem um volume relevante de vendas, mas com média de avaliação (review) baixa.  
O objetivo é identificar padrões e possíveis causas para esse comportamento.

**Fonte dos dados:** [Olist E-Commerce Public Dataset - Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)

---
## 1. Importação das bibliotecas

In [49]:
import pandas as pd
from pathlib import Path

---
## 2. Carregamento das tabelas

Carregamos apenas as tabelas necessárias para a análise:
- **order_items**: relaciona pedidos com vendedores e produtos
- **order_reviews**: contém as notas dadas pelos clientes
- **orders**: status e datas dos pedidos
- **sellers**: informações dos vendedores (cidade/estado)

In [50]:
# Carregando as tabelas do dataset Olist
data_dir = Path("data")

reviews = pd.read_csv(data_dir / "olist_order_reviews_dataset.csv")
order_items = pd.read_csv(data_dir / "olist_order_items_dataset.csv")
orders = pd.read_csv(data_dir / "olist_orders_dataset.csv")
sellers = pd.read_csv(data_dir / "olist_sellers_dataset.csv")
products = pd.read_csv(data_dir / "olist_products_dataset.csv")

print('Linhas carregadas:')
print(f'  orders:      {len(orders):>7,}')
print(f'  order_items: {len(order_items):>7,}')
print(f'  reviews:     {len(reviews):>7,}')
print(f'  products:    {len(products):>7,}')
print(f'  sellers:     {len(sellers):>7,}')

Linhas carregadas:
  orders:       99,441
  order_items: 112,650
  reviews:      99,224
  products:     32,951
  sellers:       3,095


---
## 3. Limpeza das tabelas

Cada tabela é limpa separadamente para facilitar a leitura e replicação.

In [51]:
# ── Limpeza: orders ──────────────────────────────────────────────────────────
# Mantemos apenas pedidos com status 'delivered' (entregues ao cliente)
# Pedidos cancelados, em processamento ou não entregues distorcem a análise de satisfação

orders_clean = orders[orders['order_status'] == 'delivered'].copy()

# Remove linhas sem data de entrega registrada (entrega não confirmada)
orders_clean = orders_clean.dropna(subset=['order_delivered_customer_date'])

print(f'orders original:  {len(orders):,} linhas')
print(f'orders limpo:     {len(orders_clean):,} linhas')

orders original:  99,441 linhas
orders limpo:     96,470 linhas


In [52]:
# ── Limpeza: reviews ─────────────────────────────────────────────────────────
# Remove linhas sem review_score (avaliação não registrada)
reviews_clean = reviews.dropna(subset=['review_score']).copy()

# Garante que review_score seja inteiro entre 1 e 5
reviews_clean = reviews_clean[reviews_clean['review_score'].between(1, 5)]

# Remove duplicatas: mesmo order_id com review duplicada (mantém a primeira)
reviews_clean = reviews_clean.drop_duplicates(subset='order_id', keep='first')

print(f'reviews original: {len(reviews):,} linhas')
print(f'reviews limpo:    {len(reviews_clean):,} linhas')

reviews original: 99,224 linhas
reviews limpo:    98,673 linhas


In [53]:
# ── Limpeza: order_items ─────────────────────────────────────────────────────
# Remove itens sem seller_id (não é possível identificar o vendedor)
items_clean = order_items.dropna(subset=['seller_id']).copy()

# Remove preços zerados ou negativos (registros inválidos)
items_clean = items_clean[items_clean['price'] > 0]

print(f'order_items original: {len(order_items):,} linhas')
print(f'order_items limpo:    {len(items_clean):,} linhas')

order_items original: 112,650 linhas
order_items limpo:    112,650 linhas


In [54]:
# ── Limpeza: sellers ─────────────────────────────────────────────────────────
# Remove vendedores sem cidade ou estado (dados incompletos de localização)
sellers_clean = sellers.dropna(subset=['seller_city', 'seller_state']).copy()

print(f'sellers original: {len(sellers):,} linhas')
print(f'sellers limpo:    {len(sellers_clean):,} linhas')

sellers original: 3,095 linhas
sellers limpo:    3,095 linhas


---
## 4. Montagem da base analítica de vendedores

Cruzamos as tabelas para calcular, por vendedor:
- Número total de pedidos entregues
- Média de avaliação (review_score)
- Receita total gerada

Em seguida, filtramos os vendedores que têm **muitas vendas e baixa nota**.

In [55]:
# Junta order_items com os pedidos entregues
items_delivered = items_clean.merge(
    orders_clean[['order_id']],
    on='order_id',
    how='inner'
)

# Junta com as avaliações
items_with_review = items_delivered.merge(
    reviews_clean[['order_id', 'review_score']],
    on='order_id',
    how='left'   # left para manter pedidos sem review (serão ignorados nas médias)
)

# Agrupamento por vendedor: contagem de pedidos únicos, média de nota e receita total
seller_stats = (
    items_with_review
    .groupby('seller_id')
    .agg(
        total_pedidos  = ('order_id',      'nunique'),
        media_nota     = ('review_score',  'mean'),
        receita_total  = ('price',         'sum')
    )
    .reset_index()
)

# Junta com estado/cidade dos vendedores
seller_stats = seller_stats.merge(
    sellers_clean[['seller_id', 'seller_state', 'seller_city']],
    on='seller_id',
    how='left'
)

print(f'Total de vendedores com dados: {len(seller_stats):,}')
seller_stats.head()

Total de vendedores com dados: 2,970


,seller_id,total_pedidos,media_nota,receita_total,seller_state,seller_city
0,0015a82c2db000af6aaaf3ae2ecb0532,3,3.666667,2685.00,SP,santo andre
1,001cca7ae9ae17fb1caed9dfb1094831,195,3.965368,24487.03,ES,cariacica
2,002100f778ceb8431b7a1020ff7ab48f,50,4.037037,1216.60,SP,franca
3,003554e2dce176b5555353e4f3555ac8,1,5.000000,120.00,GO,goiania
4,004c9cd9d87a3c30c522c48c4fc07416,156,4.132530,19569.73,SP,ibitinga


In [59]:
# ── Filtro: vendedores com vendas mínimas e nota baixa ────────────────────────
#
# Critérios adotados (ajustáveis conforme necessidade):
#   - Mínimo de 3 pedidos   → mantém apenas vendedores com amostra mínima
#   - Média de nota <= 2.0  → abaixo da média geral do dataset (~ 4.0)
#
MIN_PEDIDOS = 3
MAX_NOTA    = 2

sellers_problema = seller_stats[
    (seller_stats['total_pedidos'] >= MIN_PEDIDOS) &
    (seller_stats['media_nota']    <= MAX_NOTA)
].sort_values('media_nota')

print(f'Vendedores com >= {MIN_PEDIDOS} pedidos e nota <= {MAX_NOTA}: {len(sellers_problema)}')
sellers_problema.head(10)

Vendedores com >= 3 pedidos e nota <= 2: 10


,seller_id,total_pedidos,media_nota,receita_total,seller_state,seller_city
875,4e326052e5dbba8adcd512f3450a307e,3,1.666667,570.00,SP,sao paulo
542,30c7f28fd3a5897b2c82d152bb760c17,4,1.666667,469.58,RJ,niteroi
1583,8bd0e3abda539b9479c4b44a691be1ec,6,1.750000,425.75,RS,tres de maio
1116,633ecdf879b94b5337cca303328e4a25,6,1.800000,1037.60,SP,sao paulo
1453,7fdb0720c8d7c9075538b365dc8c3a22,5,1.833333,656.00,PR,toledo
905,5151aea44289d6c6b090ee31c2132508,7,1.833333,1315.30,SP,marilia
290,1b65c144b17e607c0f37f10bb7dfec8d,5,2.000000,1109.50,SC,rio do sul
402,2528744c5ef5d955adc318720a94d2e7,3,2.000000,1647.00,SP,jundiai
1565,8a43128d7f9a3db592b866e6861a6cce,3,2.000000,990.60,SP,sao paulo
1857,a53f60f05f822b5c621a50ff6a5cb362,3,2.000000,345.50,MG,uberlandia


---
## 5. Tratamento de outliers extremos

Nesta etapa removemos valores **muito fora da realidade** para reduzir ruído e distorções no resultado final.

In [62]:
# Função utilitária para remover outliers extremos por IQR
# fator=5 remove apenas pontos muito distantes (mais conservador que o padrão 1.5)
def filtrar_outlier_extremo_iqr(df, coluna, fator=5.0, limite_inferior_zero=False):
    base = df.copy()
    serie = pd.to_numeric(base[coluna], errors='coerce')

    base = base[serie.notna()].copy()
    serie = pd.to_numeric(base[coluna], errors='coerce')

    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1

    lim_inf = q1 - fator * iqr
    lim_sup = q3 + fator * iqr

    if limite_inferior_zero:
        lim_inf = max(0, lim_inf)

    mask = serie.between(lim_inf, lim_sup)
    removidos = int((~mask).sum())

    return base[mask].copy(), lim_inf, lim_sup, removidos


# Relatório de limpeza extrema
relatorio_outliers = []

# 1) Outliers extremos em itens (preço e frete)
items_ext = items_clean.copy()
items_ext['price'] = pd.to_numeric(items_ext['price'], errors='coerce')
items_ext['freight_value'] = pd.to_numeric(items_ext['freight_value'], errors='coerce')
items_ext = items_ext[(items_ext['price'] > 0) & (items_ext['freight_value'] >= 0)].copy()

for coluna in ['price', 'freight_value']:
    antes = len(items_ext)
    items_ext, lim_inf, lim_sup, removidos = filtrar_outlier_extremo_iqr(
        items_ext,
        coluna,
        fator=5.0,
        limite_inferior_zero=True,
    )
    relatorio_outliers.append({
        'tabela': 'items_ext',
        'coluna': coluna,
        'linhas_antes': antes,
        'linhas_depois': len(items_ext),
        'removidos': removidos,
        'limite_inferior': round(lim_inf, 2),
        'limite_superior': round(lim_sup, 2),
    })

# 2) Outliers extremos em prazo de entrega
orders_ext = orders_clean.copy()
orders_ext['order_purchase_timestamp'] = pd.to_datetime(orders_ext['order_purchase_timestamp'], errors='coerce')
orders_ext['order_delivered_customer_date'] = pd.to_datetime(orders_ext['order_delivered_customer_date'], errors='coerce')
orders_ext = orders_ext.dropna(subset=['order_purchase_timestamp', 'order_delivered_customer_date']).copy()

orders_ext['dias_entrega'] = (
    orders_ext['order_delivered_customer_date'] - orders_ext['order_purchase_timestamp']
).dt.days

# Faixa lógica inicial: descarta entrega negativa ou absurdamente longa
orders_ext = orders_ext[(orders_ext['dias_entrega'] >= 0) & (orders_ext['dias_entrega'] <= 365)].copy()

antes = len(orders_ext)
orders_ext, lim_inf, lim_sup, removidos = filtrar_outlier_extremo_iqr(
    orders_ext,
    'dias_entrega',
    fator=5.0,
    limite_inferior_zero=True,
)
relatorio_outliers.append({
    'tabela': 'orders_ext',
    'coluna': 'dias_entrega',
    'linhas_antes': antes,
    'linhas_depois': len(orders_ext),
    'removidos': removidos,
    'limite_inferior': round(lim_inf, 2),
    'limite_superior': round(lim_sup, 2),
})

# 3) Reconstroi base de vendedores com dados já limpos de extremos
items_delivered = items_ext.merge(
    orders_ext[['order_id']],
    on='order_id',
    how='inner',
)

items_with_review = items_delivered.merge(
    reviews_clean[['order_id', 'review_score']],
    on='order_id',
    how='left',
)

seller_stats = (
    items_with_review
    .groupby('seller_id')
    .agg(
        total_pedidos=('order_id', 'nunique'),
        media_nota=('review_score', 'mean'),
        receita_total=('price', 'sum'),
    )
    .reset_index()
)

seller_stats = seller_stats.merge(
    sellers_clean[['seller_id', 'seller_state', 'seller_city']],
    on='seller_id',
    how='left',
)

# 4) Outliers extremos no agregado de vendedores (pedido e receita)
for coluna in ['total_pedidos', 'receita_total']:
    antes = len(seller_stats)
    seller_stats, lim_inf, lim_sup, removidos = filtrar_outlier_extremo_iqr(
        seller_stats,
        coluna,
        fator=5.0,
        limite_inferior_zero=True,
    )
    relatorio_outliers.append({
        'tabela': 'seller_stats',
        'coluna': coluna,
        'linhas_antes': antes,
        'linhas_depois': len(seller_stats),
        'removidos': removidos,
        'limite_inferior': round(lim_inf, 2),
        'limite_superior': round(lim_sup, 2),
    })

# 5) Reaplica filtro de interesse após limpeza extrema
sellers_problema = seller_stats[
    (seller_stats['total_pedidos'] >= MIN_PEDIDOS) &
    (seller_stats['media_nota'] <= MAX_NOTA)
].sort_values('media_nota')

relatorio_outliers_df = pd.DataFrame(relatorio_outliers)

print('Limpeza extrema concluida.')
print(f'Vendedores com >= {MIN_PEDIDOS} pedidos e nota <= {MAX_NOTA} apos limpeza: {len(sellers_problema)}')
display(relatorio_outliers_df)

Limpeza extrema concluida.
Vendedores com >= 3 pedidos e nota <= 2 apos limpeza: 10


,tabela,coluna,linhas_antes,linhas_depois,removidos,limite_inferior,limite_superior
0,items_ext,price,112650,110342,2308,0,609.90
1,items_ext,freight_value,110342,108084,2258,0,60.00
2,orders_ext,dias_entrega,96470,96182,288,0,60.00
3,seller_stats,total_pedidos,2855,2693,162,0,122.00
4,seller_stats,receita_total,2693,2622,71,0,10749.14


---
## 6. Resultado final após limpeza extrema

Resumo descritivo da base final e dos vendedores com baixa avaliação após remover outliers muito fora da realidade.

In [63]:
# Estatísticas descritivas da base final (sem gráficos)
print('=== BASE FINAL LIMPA (todos os vendedores) ===')
print(seller_stats[['total_pedidos', 'media_nota', 'receita_total']].describe().round(2))

print('\n=== VENDEDORES COM BAIXA NOTA (após limpeza extrema) ===')
if sellers_problema.empty:
    print('Nenhum vendedor encontrado com o critério atual.')
else:
    print(sellers_problema[['total_pedidos', 'media_nota', 'receita_total']].describe().round(2))

print('\n=== AMOSTRA DOS VENDEDORES COM BAIXA NOTA ===')
display(
    sellers_problema[['seller_id', 'seller_state', 'seller_city', 'total_pedidos', 'media_nota', 'receita_total']]
    .sort_values(['media_nota', 'total_pedidos'], ascending=[True, False])
    .head(20)
)

print('\n=== RELATÓRIO DE LINHAS REMOVIDAS POR OUTLIER EXTREMO ===')
display(relatorio_outliers_df)

=== BASE FINAL LIMPA (todos os vendedores) ===
       total_pedidos  media_nota  receita_total
count        2622.00     2618.00        2622.00
mean           13.63        4.17        1442.41
std            20.04        0.82        2006.67
min             1.00        1.00           6.50
25%             2.00        3.89         179.92
50%             6.00        4.33         599.00
75%            15.75        4.75        1758.97
max           122.00        5.00       10607.00

=== VENDEDORES COM BAIXA NOTA (após limpeza extrema) ===
       total_pedidos  media_nota  receita_total
count          10.00       10.00          10.00
mean            4.50        1.85         856.68
std             1.51        0.14         430.61
min             3.00        1.67         345.50
25%             3.00        1.76         494.69
50%             4.50        1.83         823.30
75%             5.75        2.00        1091.53
max             7.00        2.00        1647.00

=== AMOSTRA DOS VENDEDORES COM

,seller_id,seller_state,seller_city,total_pedidos,media_nota,receita_total
542,30c7f28fd3a5897b2c82d152bb760c17,RJ,niteroi,4,1.666667,469.58
875,4e326052e5dbba8adcd512f3450a307e,SP,sao paulo,3,1.666667,570.00
1583,8bd0e3abda539b9479c4b44a691be1ec,RS,tres de maio,6,1.750000,425.75
1116,633ecdf879b94b5337cca303328e4a25,SP,sao paulo,6,1.800000,1037.60
905,5151aea44289d6c6b090ee31c2132508,SP,marilia,7,1.833333,1315.30
1453,7fdb0720c8d7c9075538b365dc8c3a22,PR,toledo,5,1.833333,656.00
290,1b65c144b17e607c0f37f10bb7dfec8d,SC,rio do sul,5,2.000000,1109.50
402,2528744c5ef5d955adc318720a94d2e7,SP,jundiai,3,2.000000,1647.00
1565,8a43128d7f9a3db592b866e6861a6cce,SP,sao paulo,3,2.000000,990.60
1857,a53f60f05f822b5c621a50ff6a5cb362,MG,uberlandia,3,2.000000,345.50



=== RELATÓRIO DE LINHAS REMOVIDAS POR OUTLIER EXTREMO ===


,tabela,coluna,linhas_antes,linhas_depois,removidos,limite_inferior,limite_superior
0,items_ext,price,112650,110342,2308,0,609.90
1,items_ext,freight_value,110342,108084,2258,0,60.00
2,orders_ext,dias_entrega,96470,96182,288,0,60.00
3,seller_stats,total_pedidos,2855,2693,162,0,122.00
4,seller_stats,receita_total,2693,2622,71,0,10749.14
